# Gate C, consumed-lock and history lineage

September 4, 2026. Source-based offline diagnosis for August 15–September 3. Reads captured SQL/API evidence and the preserved historical Gate C file. No network, database, history, notification or production writes. This notebook executes `reproduce.py`; that source contains the complete joins and predicates and regenerates only this packet’s derived files. The linkage preview grants no prospective credit.

In [1]:
from pathlib import Path
import contextlib, io, json, gzip, runpy
root=Path.cwd()
assert (root/'reproduce.py').is_file()
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path(str(root/'reproduce.py'),run_name='__main__')
a=json.loads((root/'analysis.json').read_text())
print('Offline reconstruction completed:',a['rebuild'])

Offline reconstruction completed: {'dates': 20, 'diagnostics': {'archive_outcome_reconciliation': {'ambiguous_markets': 0, 'recovered_markets': 0}}, 'duplicates': 0, 'reconciliation': {'graded_pick_rows': 330, 'matched_pick_rows': 330, 'unique_side_fallback_matches': 0, 'unmatched_examples': [], 'unmatched_pick_rows': 0}, 'rows': 660, 'tracked_graded': 351}


## Historical manifest correction

The earlier assessment checked LF normalization only. Exact in-memory CRLF reconstruction matches both Windows-issued manifest hashes, without changing a single historical file. This does not extend the June 16 data window.

In [2]:
p=a['gate_c_provenance']
assert p['rows']==2070 and p['row_values_identical']
assert p['reconstructed_windows_crlf_sha256']==p['manifest_sha256']
assert p['summary_crlf_sha256']==p['manifest_summary_sha256']
print({k:v for k,v in p.items() if k!='source_window'})
print('Actual data window:',p['source_window']['start_date'],p['source_window']['end_date'])

{'lf_sha256': '6f5745a290b12ab3976a81f5231084eb8619adc855384d1107187ad5c51ff66b', 'manifest_sha256': 'e8c7c6d53ca51610213d1b168af58e831f5f8e079f2e28f88944856c3316a0da', 'manifest_summary_sha256': '6fec2e22c0e1da27aefe850d4fd2f43fe099f5fcf092da67d741199961183d88', 'reconstructed_windows_crlf_sha256': 'e8c7c6d53ca51610213d1b168af58e831f5f8e079f2e28f88944856c3316a0da', 'row_values_identical': True, 'rows': 2070, 'summary_crlf_sha256': '6fec2e22c0e1da27aefe850d4fd2f43fe099f5fcf092da67d741199961183d88', 'summary_lf_sha256': '5528801514d2fad35f256e53747758d4ab961d64da3be8c8c1c56567acda011a'}
Actual data window: 2026-04-28 2026-06-16


## Denominators and strict linkage

A history match is not proof of consumption. Keep the builder’s history-based reconciliation separate from the operational-lock denominator. Exact book disagreements remain exclusions.

In [3]:
c=a['consumption'];j=a['lock_join']
assert c['operational_locks']==330 and c['consumed']==c['closed_locked_history']==329
assert len(c['missing_history'])==1 and c['missing_history'][0]['consumed_at'] is None
assert a['rebuild']['reconciliation']['matched_pick_rows']==330
assert j['exact_matches']+len(j['exclusions'])==a['rebuild']['tracked_graded']==351
from collections import Counter
print(c)
print('Exact matches:',j['exact_matches'],'; exclusions:',dict(Counter(reason for r in j['exclusions'] for reason in r['reasons'])))

{'closed_locked_history': 329, 'consumed': 329, 'missing_history': [{'consumed_at': None, 'identity': ['2026-08-19', 'kumar rocker', 'under']}], 'operational_locks': 330}
Exact matches: 314 ; exclusions: {'bet_time_book_mismatch': 15, 'missing_or_ambiguous_operational_lock': 22}


## Selective-LEAN eligibility and timing

The four-field selector and fingerprint are unchanged. This is a schema preview, not the complete frozen-baseline audit. A lock-only join cannot create missing market agreement, official-provider reader support or prospective preclose evidence. Final-archive and frozen input matches also differ.

In [4]:
s=a['selective_lean'];f=s['frozen_comparison']
assert s['matched_candidates']==23
assert s['pass_missing_inputs_before']==s['pass_missing_inputs_after_lock_only']==s['formal_credit_granted']==0
assert f['overlap']==21 and len(f['frozen_only'])==len(f['archive_only'])==2
assert f['preclose_freshness']=={'pending':23}
print('Remaining gaps after lock-only preview:',s['missing_after_lock_only_preview'])
print('Frozen versus archive:',f)

Remaining gaps after lock-only preview: {'bet_timing_window=pre_30': 1, 'market_agreement_label|market_agreement': 23, 'operational_lock_consumed_at|consumed_at|lock_consumed_at': 3, 'operational_lock_id|lock_id|lock_key|operational_lock_key': 3, 'operational_lock_source_artifact_path|lock_source_artifact_path': 3, 'preclose_clv_proxy_label|preclose_clv_proxy': 23, 'provider|live_display_provider|odds_source': 23}
Frozen versus archive: {'archive_only': [['2026-08-21', 'randy vasquez', 'under'], ['2026-09-02', 'justin hagenman', 'over']], 'archive_rule_matches': 23, 'frozen_only': [['2026-08-29', 'erick fedde', 'over'], ['2026-08-30', 'mason adams', 'over']], 'frozen_rule_matches': 23, 'overlap': 21, 'preclose_freshness': {'pending': 23}, 'preclose_labels': {'None': 23}, 'preclose_reason_codes': {'exact_ladder_missing': 8, 'official_provider_immature': 8, 'provider_run_page_cap_reached': 14, 'same_checkpoint_alternate_line_ambiguity': 13, 'snapshot_page_cap_reached': 23}}


## Kumar Rocker exception

MLB game 822860 identifies the actual outcome. The operational lock remains unconsumed, so the existing repair contract rejects it. Do not convert this hypothetical quote outcome into a repaired official pick.

In [5]:
k=json.loads((root/'kumar-source.json').read_text())
b=json.loads((root/'mlb-kumar-boxscore.json').read_text())
assert k['operational_lock']['consumed_at'] is None and k['history']['matches'] is None
assert b['player']['person']['id']==677958 and b['player']['stats']['pitching']['strikeOuts']==3
assert k['operational_lock']['locked_k_line']==4.5 and k['operational_lock']['locked_odds']==-138
from scripts.repair_missing_locked_picks import _validate_lock
try:
    _validate_lock(k['operational_lock'])
except ValueError as error:
    assert 'unconsumed' in str(error)
    print('Existing repair correctly rejects:',str(error))
else:
    raise AssertionError('Unconsumed lock unexpectedly accepted')
print('Side evidence only: 3 K; quoted UNDER would return',round(100/138,6),'u. No history row written.')

Existing repair correctly rejects: expected exactly one consumed lock; lock is unconsumed
Side evidence only: 3 K; quoted UNDER would return 0.724638 u. No history row written.


## Transport and interpretation limits

Stored artifact hashes identify issuer provenance. Re-serialized served JSON does not reproduce them. The separately captured August 30 SQL payload compares semantically equal to the API payload, but this does not certify every original issuer byte sequence.

In [6]:
t=json.loads((root/'transport-sample.json').read_text())
served=json.loads(gzip.decompress((root/'served-artifacts.json.gz').read_bytes()))
assert json.loads(t['payload_text'])==next(r['payload'] for r in served if r['key']=='2026-08-30')
print('SQL/API August 30 values equal; issuer hash differences remain explicit.')
print('All notebook assertions passed.')

SQL/API August 30 values equal; issuer hash differences remain explicit.
All notebook assertions passed.
